# Detection → Structured Event Pipeline (Colab)

## Goal
Run YOLOv8 on drone imagery/video and convert detections into structured "event" records.
Then perform basic query-style analytics (counts by class, counts by time window, peak activity).

This notebook reproduces the key system idea used by real-world video analytics systems:
**object detections become structured metadata that is queryable.**

Outputs (events.jsonl, events.csv) are saved to Google Drive for GitHub submission.

# Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Install

In [2]:
!pip -q install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.5 MB/s eta 0:00:00


#Imports

In [3]:
import os
import json
import time
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import pandas as pd

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# GPU Check

In [4]:
!nvidia-smi

Sun Feb  8 16:29:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
DEVICE = 0  # GPU if available

# Paths

In [15]:
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Capstone/Agentic_Drone_OD" #This should be changed to your own root

# Dataset root
DATA_ROOT = DRIVE_PROJECT_ROOT

# Where outputs will go
OUT_DIR = os.path.join(DRIVE_PROJECT_ROOT, "reproductions/02_detection_to_event_pipeline")
os.makedirs(OUT_DIR, exist_ok=True)

#VIDEO_PATH = os.path.join(DRIVE_PROJECT_ROOT, "sample_videos", "drone_clip.mp4")

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)
#print("VIDEO_PATH exists?", os.path.exists(VIDEO_PATH))

DATA_ROOT: /content/drive/MyDrive/Capstone/Agentic_Drone_OD
OUT_DIR: /content/drive/MyDrive/Capstone/Agentic_Drone_OD/reproductions/02_detection_to_event_pipeline


In [8]:
#MODE = "video" if os.path.exists(VIDEO_PATH) else "images"
MODE = "images"
print("MODE:", MODE)

MODE: images


# Load Model

In [9]:
MODEL_WEIGHTS = "yolov8n.pt"  # or path to your fine-tuned weights in Drive

model = YOLO(MODEL_WEIGHTS)
print("Loaded:", MODEL_WEIGHTS)

Loaded: yolov8n.pt


# Event Schema

In [10]:
def build_event(
    source: str,
    source_id: str,
    frame_index: int,
    timestamp_sec: Optional[float],
    class_id: int,
    class_name: str,
    confidence: float,
    bbox_xyxy: List[float],
    img_w: int,
    img_h: int
) -> Dict[str, Any]:
    x1, y1, x2, y2 = bbox_xyxy
    # normalized (0-1)
    xywhn = [
        ((x1 + x2) / 2) / img_w,
        ((y1 + y2) / 2) / img_h,
        (x2 - x1) / img_w,
        (y2 - y1) / img_h,
    ]
    return {
        "source": source,                 # "video" or "image"
        "source_id": source_id,           # filename or video path
        "frame_index": frame_index,       # frame # for video, 0 for image
        "timestamp_sec": timestamp_sec,   # None for images
        "class_id": int(class_id),
        "class_name": class_name,
        "confidence": float(confidence),
        "bbox_xyxy": [float(x) for x in bbox_xyxy],
        "bbox_xywhn": [float(x) for x in xywhn],
        "img_width": int(img_w),
        "img_height": int(img_h),
    }

# Run for Video (Video -> Events)

In [ ]:
events: List[Dict[str, Any]] = []

if MODE == "video":
    results = model.predict(
        source=VIDEO_PATH,
        conf=0.25,
        iou=0.65,
        imgsz=640,
        device=DEVICE,
        stream=True,   # important: yields per-frame results
        verbose=False
    )

    frame_idx = 0
    start = time.time()

    for r in results:
        # r is a Results object for one frame
        # best-effort timestamp: use fps if available
        # Ultralytics may provide r.path, r.orig_shape, r.boxes
        img_h, img_w = r.orig_shape  # (h, w)
        source_id = os.path.basename(r.path) if getattr(r, "path", None) else os.path.basename(VIDEO_PATH)

        # Attempt to compute timestamp from frame index + fps if metadata exists
        # If fps not known, leave None
        timestamp_sec = None

        if r.boxes is not None and len(r.boxes) > 0:
            boxes = r.boxes
            for b in boxes:
                cls_id = int(b.cls.item()) if hasattr(b.cls, "item") else int(b.cls)
                conf = float(b.conf.item()) if hasattr(b.conf, "item") else float(b.conf)
                xyxy = b.xyxy[0].tolist() if hasattr(b.xyxy[0], "tolist") else list(b.xyxy[0])

                class_name = model.names.get(cls_id, str(cls_id)) if hasattr(model, "names") else str(cls_id)

                events.append(build_event(
                    source="video",
                    source_id=source_id,
                    frame_index=frame_idx,
                    timestamp_sec=timestamp_sec,
                    class_id=cls_id,
                    class_name=class_name,
                    confidence=conf,
                    bbox_xyxy=xyxy,
                    img_w=img_w,
                    img_h=img_h
                ))

        frame_idx += 1

    print("Frames processed:", frame_idx)
    print("Events captured:", len(events))
    print("Elapsed seconds:", round(time.time() - start, 2))

# Run for images (Images -> Events)

In [18]:
import glob
import os

events = []

if MODE == "images":
    # Point directly to your validation images folder
    val_dir = os.path.join(DATA_ROOT, "val", "images")

    img_paths = (
        glob.glob(os.path.join(val_dir, "*.jpg")) +
        glob.glob(os.path.join(val_dir, "*.jpeg")) +
        glob.glob(os.path.join(val_dir, "*.png"))
    )

    # Limit for fast reproduction
    img_paths = sorted(img_paths)[:50]

    if len(img_paths) == 0:
        raise RuntimeError(f"No images found in {val_dir}")

    results = model.predict(
        source=img_paths,
        conf=0.25,
        iou=0.65,
        imgsz=640,
        device=DEVICE,
        stream=True,
        verbose=False
    )

    for r in results:
        img_h, img_w = r.orig_shape
        source_id = os.path.basename(r.path) if getattr(r, "path", None) else "image"

        if r.boxes is not None and len(r.boxes) > 0:
            for b in r.boxes:
                cls_id = int(b.cls.item()) if hasattr(b.cls, "item") else int(b.cls)
                conf = float(b.conf.item()) if hasattr(b.conf, "item") else float(b.conf)
                xyxy = b.xyxy[0].tolist() if hasattr(b.xyxy[0], "tolist") else list(b.xyxy[0])

                class_name = (
                    model.names.get(cls_id, str(cls_id))
                    if hasattr(model, "names")
                    else str(cls_id)
                )

                events.append(build_event(
                    source="image",
                    source_id=source_id,
                    frame_index=0,
                    timestamp_sec=None,
                    class_id=cls_id,
                    class_name=class_name,
                    confidence=conf,
                    bbox_xyxy=xyxy,
                    img_w=img_w,
                    img_h=img_h
                ))

    print("Images processed:", len(img_paths))
    print("Events captured:", len(events))

Images processed: 50
Events captured: 674


# Save to drive

In [19]:
events_jsonl = os.path.join(OUT_DIR, "events.jsonl")
events_csv = os.path.join(OUT_DIR, "events.csv")

# JSON Lines for easy streaming + logging
with open(events_jsonl, "w") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")

# CSV for quick analysis
df = pd.DataFrame(events)
df.to_csv(events_csv, index=False)

print("✅ Saved events:")
print("JSONL:", events_jsonl)
print("CSV  :", events_csv)
print("Rows :", len(df))

✅ Saved events:
JSONL: /content/drive/MyDrive/Capstone/Agentic_Drone_OD/reproductions/02_detection_to_event_pipeline/events.jsonl
CSV  : /content/drive/MyDrive/Capstone/Agentic_Drone_OD/reproductions/02_detection_to_event_pipeline/events.csv
Rows : 674


# Query Tests

In [20]:
counts_by_class = df.groupby("class_name").size().sort_values(ascending=False)
print("Detections by class:")
print(counts_by_class.head(25))

Detections by class:
class_name
person           397
car              232
cell phone         7
truck              6
bus                6
motorcycle         4
umbrella           4
bicycle            4
boat               3
cow                3
bench              2
frisbee            1
book               1
sports ball        1
potted plant       1
traffic light      1
suitcase           1
dtype: int64


In [21]:
high_conf = df[df["confidence"] >= 0.6]
counts_high_conf = high_conf.groupby("class_name").size().sort_values(ascending=False)

print("High-confidence detections (conf>=0.6) by class:")
print(counts_high_conf.head(25))

High-confidence detections (conf>=0.6) by class:
class_name
person        120
car            60
cell phone      2
bicycle         1
dtype: int64


In [ ]:
# Video Only
if MODE == "video":
    # If timestamp is None, fallback to frame bins
    if df["timestamp_sec"].isna().all():
        df["time_bin"] = (df["frame_index"] // 150)  # roughly "chunks" of frames
        peak = df.groupby("time_bin").size().sort_values(ascending=False).head(10)
        print("Peak bins (by frame chunk):")
        print(peak)
    else:
        df["time_bin"] = (df["timestamp_sec"] // 10)  # 10-second bins
        peak = df.groupby("time_bin").size().sort_values(ascending=False).head(10)
        print("Peak 10-second bins:")
        print(peak)
else:
    print("MODE is images; skipping time-window analytics.")

## What this reproduction demonstrates

This notebook reproduces a common video analytics pattern used in real-world systems:
1) Run an object detector (YOLOv8) on drone imagery/video.
2) Convert frame-level detections into structured event records (class, bbox, confidence, time/frame index).
3) Persist events to machine-readable formats (JSONL/CSV) that support downstream analysis and querying.